In [ ]:
%pip install tensorflow tensorflow-hub==0.11.0 kipoiseq datasets scipy matplotlib seaborn scikit-learn tqdm

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import pandas as pd
from typing import List, Optional, Union, Tuple, Dict
import logging
from pathlib import Path
import kipoiseq
from kipoiseq import Interval
import functools
from tqdm import tqdm

from datasets import load_dataset

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

from time import time

import json

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class EnformerEmbeddingExtractor:
    """
    A class for extracting embeddings from DNA/RNA sequences using the original Enformer model.
    
    Attributes:
        model: The loaded Enformer model from TensorFlow Hub
        sequence_length: Expected input sequence length
        crop_size: Output crop size for predictions
        target_length: Length of target predictions
    """
    
    def __init__(self, 
                 model_path: str = 'https://tfhub.dev/deepmind/enformer/1',
                 use_gpu: bool = True,
                 mixed_precision: bool = False):
        """
        Initialize the Enformer embedding extractor.
        
        Args:
            model_path: Path to the Enformer model (default: TensorFlow Hub URL)
            use_gpu: Whether to use GPU if available
            mixed_precision: Whether to use mixed precision for faster inference
        """
        self.model_path = model_path
        self.sequence_length = 393_216
        self.crop_size = 8192  # Crop size for target region
        self.target_length = 896
        self.num_channels   = 5313
        
        # Configure TensorFlow
        self._configure_tensorflow(use_gpu, mixed_precision)
        
        # Load the model
        self.model = None
        self._load_model()
        
        # Initialize sequence processing utilities
        self.transform = self._get_transform()
        
    def _configure_tensorflow(self, use_gpu: bool, mixed_precision: bool):
        """Configure TensorFlow settings."""
        if use_gpu:
            physical_devices = tf.config.experimental.list_physical_devices('GPU')
            if physical_devices:
                try:
                    for device in physical_devices:
                        tf.config.experimental.set_memory_growth(device, True)
                    logger.info(f"Found {len(physical_devices)} GPU(s)")
                except RuntimeError as e:
                    logger.warning(f"GPU configuration failed: {e}")
        
        if mixed_precision:
            policy = tf.keras.mixed_precision.Policy('mixed_float16')
            tf.keras.mixed_precision.set_global_policy(policy)
            logger.info("Mixed precision enabled")
    
    def _load_model(self):
        """Load the Enformer model from TensorFlow Hub."""
        try:
            logger.info(f"Loading Enformer model from {self.model_path}")
            self.model = hub.load(self.model_path).model
            logger.info("Model loaded successfully")
        except Exception as e:
            logger.error(f"Failed to load model: {e}")
            raise
    
    def _get_transform(self):
        """Get the transformation function for sequence processing."""
        return kipoiseq.transforms.Compose([
            kipoiseq.transforms.functional.one_hot_dna
        ])
    
    def _validate_sequence(self, sequence: str) -> str:
        """
        Validate and preprocess a DNA/RNA sequence.
        
        Args:
            sequence: Input DNA/RNA sequence string
            
        Returns:
            Processed sequence string
        """

        sequence = sequence.upper().replace('U', 'T')
        
        valid_nucleotides = set('ATCGN')
        if not set(sequence).issubset(valid_nucleotides):
            invalid_chars = set(sequence) - valid_nucleotides
            raise ValueError(f"Invalid nucleotides found: {invalid_chars}")
        
        return sequence
    
    def _pad_or_crop_sequence(self, sequence: str) -> str:
        """
        Pad or crop sequence to the required length.
        
        Args:
            sequence: Input sequence
            
        Returns:
            Sequence of exactly self.sequence_length
        """
        if len(sequence) > self.sequence_length:
            # Crop from center
            start = (len(sequence) - self.sequence_length) // 2
            return sequence[start:start + self.sequence_length]
        elif len(sequence) < self.sequence_length:
            # Pad with N's (center the sequence)
            pad_length = self.sequence_length - len(sequence)
            left_pad = pad_length // 2
            right_pad = pad_length - left_pad
            return 'N' * left_pad + sequence + 'N' * right_pad
        else:
            return sequence
    
    def _sequence_to_tensor(self, sequence: str) -> tf.Tensor:
        """
        Convert a sequence string to a tensor.
        
        Args:
            sequence: DNA/RNA sequence string
            
        Returns:
            Tensor of shape (1, sequence_length, 4)
        """
        # Validate and preprocess sequence
        sequence = self._validate_sequence(sequence)
        sequence = self._pad_or_crop_sequence(sequence)
        
        # Convert to one-hot encoding
        one_hot = self.transform(sequence).astype(np.float32)
        
        # Add batch dimension
        return tf.expand_dims(one_hot, axis=0)
    
    def extract_embeddings(self, 
                          sequences: List[str],
                          batch_size: int = 1,
                          return_raw: bool = False) -> Union[np.ndarray, Dict[str, np.ndarray]]:
        """
        Extract embeddings from a list of DNA/RNA sequences.
        
        Args:
            sequences: List of DNA/RNA sequence strings
            batch_size: Batch size for processing
            return_raw: If True, return raw model outputs, otherwise return processed embeddings
            
        Returns:
            If return_raw=False: numpy array of shape (n_sequences, target_length, num_channels)
            If return_raw=True: Dictionary with 'human' and 'mouse' predictions
        """
        
        logger.info(f"Processing {len(sequences)} sequences with batch size {batch_size}")
        
        all_embeddings = []
        human_predictions = []
        mouse_predictions = []
        
        # Process sequences in batches
        for i in tqdm(range(0, len(sequences), batch_size), desc="Process extracting..."):
            batch_sequences = sequences[i:i + batch_size]
            
            # Convert sequences to tensors
            batch_tensors = []
            for seq in batch_sequences:
                tensor = self._sequence_to_tensor(seq)
                batch_tensors.append(tensor)
            
            # Stack tensors into batch
            if len(batch_tensors) == 1:
                batch_input = batch_tensors[0]
            else:
                batch_input = tf.concat(batch_tensors, axis=0)
            
            # Get predictions
            try:
                predictions = self.model.predict_on_batch(batch_input)
                
                if return_raw:
                    # Store raw predictions
                    human_predictions.append(predictions['human'].numpy())
                    mouse_predictions.append(predictions['mouse'].numpy())
                else:
                    # Extract embeddings (use human predictions as default)
                    human = predictions['human'].numpy()      # shape (b, L_out, C)
                    emb   = human.mean(axis=(1,2))    # shape (b,)
                    all_embeddings.append(emb)
                    
            except Exception as e:
                logger.error(f"Error processing batch {i//batch_size + 1}: {e}")
                raise
            
            logger.info(f"Processed batch {i//batch_size + 1}/{(len(sequences) + batch_size - 1)//batch_size}")
        
        if return_raw:
            return {
                'human': np.concatenate(human_predictions, axis=0),
                'mouse': np.concatenate(mouse_predictions, axis=0)
            }
        else:
            return np.concatenate(all_embeddings, axis=0)
    
    def save_embeddings(self, 
                       embeddings: np.ndarray, 
                       output_path: str,
                       sequences: Optional[List[str]] = None):
        """
        Save embeddings to file.
        
        Args:
            embeddings: Extracted embeddings
            output_path: Output file path
            sequences: Optional list of original sequences for metadata
        """
        output_path = Path(output_path)
        
        if output_path.suffix == '.npz':
            # Save as numpy compressed format
            save_dict = {'embeddings': embeddings}
            if sequences:
                save_dict['sequences'] = sequences
            np.savez_compressed(output_path, **save_dict)
        elif output_path.suffix == '.npy':
            # Save as numpy format
            np.save(output_path, embeddings)
        else:
            raise ValueError(f"Unsupported file format: {output_path.suffix}")
        
        logger.info(f"Embeddings saved to {output_path}")
    
    def load_embeddings(self, input_path: str) -> Union[np.ndarray, Dict[str, np.ndarray]]:
        """
        Load embeddings from file.
        
        Args:
            input_path: Input file path
            
        Returns:
            Loaded embeddings
        """
        input_path = Path(input_path)
        
        if input_path.suffix == '.npz':
            data = np.load(input_path)
            return dict(data)
        elif input_path.suffix == '.npy':
            return np.load(input_path)
        else:
            raise ValueError(f"Unsupported file format: {input_path.suffix}")
    
    def get_model_info(self) -> Dict[str, Union[str, int]]:
        """
        Get information about the loaded model.
        
        Returns:
            Dictionary with model information
        """
        return {
            'model_path': self.model_path,
            'sequence_length': self.sequence_length,
            'target_length': self.target_length,
            'num_channels': self.num_channels,
            'crop_size': self.crop_size
        }

In [ ]:


ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")
train_ds, test_ds = ds['train'], ds['test']

extractor = EnformerEmbeddingExtractor()


PARAMS_LOGREG = {'max_iter': 1000, 'random_state': 42}
PATH_TO_SAVE = '.'
BATCH_SIZE = 10

# Baseline experiments (full data)
baseline = {}
for task in tqdm(set(train_ds['task']), desc='Baseline'):
    tr = train_ds.filter(lambda x, t=task: x['task']==t)
    te = test_ds.filter(lambda x, t=task: x['task']==t)
    seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
    seqs_te, y_te = te['sequence'], np.array(te['label'])

    X_tr = extractor.extract_embeddings(seqs_tr, batch_size=BATCH_SIZE)
    X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)

    clf = LogisticRegression(**PARAMS_LOGREG)
    Xf = X_tr.reshape(-1,1) if X_tr.ndim==1 or X_tr.shape[1]==1 else X_tr
    Xt = X_te.reshape(-1,1) if X_te.ndim==1 or X_te.shape[1]==1 else X_te
    clf.fit(Xf, y_tr)
    preds = clf.predict(Xt)

    baseline[task] = {
        'accuracy': float(accuracy_score(y_te, preds)),
        'f1_score': float(f1_score(y_te, preds, average='macro'))
    }
    with open(f'{PATH_TO_SAVE}/results_enformer_task-{task}_baseline.json','w') as f:
        json.dump(baseline, f, indent=4)

# Few-shot experiments
def few_shot(train, test, ks=(1,5,10,20), trials=5):
    res = {}
    rng = np.random.RandomState(42)
    for task in tqdm(set(train['task']), desc='Few-shot'):
        tr = train.filter(lambda x, t=task: x['task']==t)
        te = test.filter(lambda x, t=task: x['task']==t)
        seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
        seqs_te, y_te = te['sequence'], np.array(te['label'])
        X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)
        res[task] = {}
        for k in ks:
            accs, f1s = [], []
            for _ in range(trials):
                idxs=[]
                for lbl in np.unique(y_tr):
                    locs = np.where(y_tr==lbl)[0]
                    choice = rng.choice(locs, size=min(k,len(locs)), replace=False)
                    idxs.extend(choice.tolist())
                X_k = extractor.extract_embeddings([seqs_tr[i] for i in idxs], batch_size=BATCH_SIZE)
                y_k = y_tr[idxs]
                clf=LogisticRegression(**PARAMS_LOGREG)
                Xf = X_k.reshape(-1,1) if X_k.ndim==1 or X_k.shape[1]==1 else X_k
                Xt = X_te.reshape(-1,1) if X_te.ndim==1 or X_te.shape[1]==1 else X_te
                clf.fit(Xf, y_k)
                p=clf.predict(Xt)
                accs.append(accuracy_score(y_te,p))
                f1s.append(f1_score(y_te,p,average='macro'))
            res[task][k] = {'accuracy': float(np.mean(accs)), 'f1_score': float(np.mean(f1s))}
            
            with open(f'{PATH_TO_SAVE}/results_enformer_task-{task}_k-{k}.json','w') as f:
                json.dump(res, f, indent=4)
            
    return res

results_kshot = few_shot(train_ds, test_ds)

output = {'full': baseline, 'kshot': results_kshot, 'params': PARAMS_LOGREG}
with open(f'{PATH_TO_SAVE}/results_enformer.json','w') as f:
    json.dump(output, f, indent=4)

2025-07-18 15:53:22.226016: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-18 15:53:27.232123: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
INFO:enformer_embedding_extractor:Found 1 GPU(s)
INFO:enformer_embedding_extractor:Loading Enformer model from https://tfhub.dev/deepmind/enformer/1
INFO:absl:Using /tmp/tfhub_modules to cache modules.
INFO:absl:Downloading TF-Hub Module 'https://tfhub.dev/deepmind/enformer/1'.
INFO:absl:Downloading https://tfhub.dev/deepmind/enformer/1: 522.20MB
INFO:absl:Downloaded https://tfhub.dev/deepmind/enformer/1, Total size: 960.82MB
INFO:absl:Downloaded TF-Hub Module 'https://tfhub.dev/deepmind/enformer/1'.
2025-07-18 15:54:18.989542: I tensorflow/core/common_runti